# Sales Reporting 2019 — Mini project Python

**Mục tiêu:** phân tích dữ liệu bán hàng năm 2019 nhằm đánh giá doanh thu, thời điểm đặt hàng và xu hướng mua sản phẩm cùng nhau.

Notebook dùng thư viện chuẩn Python. Toàn bộ quy trình xử lý, phân tích và tạo báo cáo được thực hiện trong các phần dưới đây.


## 1. Đọc và kiểm tra dữ liệu
Một dòng là một sản phẩm trong đơn. Kiểm tra đủ 12 file, schema, dòng trống, tiêu đề lặp, kiểu dữ liệu và địa chỉ. Giữ cờ trùng để kiểm tra thay vì tự suy đoán đó là lỗi.


In [1]:
"""Đọc, kiểm tra và tổng hợp dữ liệu; chỉ dùng thư viện chuẩn Python."""
from __future__ import annotations

import csv
import hashlib
import itertools
import re
from collections import Counter, defaultdict
from datetime import datetime
from decimal import Decimal, InvalidOperation
from pathlib import Path

FIELDS = ['Order ID', 'Product', 'Quantity Ordered', 'Price Each', 'Order Date', 'Purchase Address']
ADDRESS = re.compile(r'^.+,\s*([^,]+),\s*([A-Z]{2})\s+\d{5}(?:-\d{4})?$')


def load_sales(data_dir: Path):
    """Giữ dòng trùng hợp lệ, đánh dấu để kiểm tra vì chưa biết khóa dòng đơn."""
    paths = sorted(data_dir.glob('sales2019_*.csv'), key=lambda p: int(p.stem.split('_')[-1]))
    if {p.name for p in paths} != {f'sales2019_{m}.csv' for m in range(1, 13)}:
        raise ValueError('Cần đúng 12 file sales2019_1.csv ... sales2019_12.csv trong thư mục đầu vào.')
    rows, rejected, manifest, seen = [], [], [], set()
    quality = Counter()
    for path in paths:
        file_counts = Counter()
        with path.open(encoding='utf-8-sig', newline='') as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != FIELDS:
                raise ValueError(f'{path.name}: cấu trúc cột không đúng. Cần {FIELDS}')
            for line, raw in enumerate(reader, 2):
                quality['raw_rows'] += 1
                file_counts['raw_rows'] += 1
                values = [str(raw.get(k) or '').strip() for k in FIELDS]
                reason = None
                if None in raw or any(raw.get(k) is None for k in FIELDS):
                    reason = 'malformed_row'
                elif not any(values):
                    reason = 'empty_row'
                elif values == FIELDS:
                    reason = 'repeated_header'
                elif not all(values):
                    reason = 'missing_value'
                if reason is None:
                    order_id, product, qty_text, price_text, date_text, address = values
                    try:
                        qty = Decimal(qty_text)
                        price = Decimal(price_text)
                        if not qty.is_finite() or not price.is_finite():
                            raise ValueError('non-finite')
                        if qty <= 0 or qty != qty.to_integral_value() or price <= 0:
                            raise ValueError('non-positive or fractional quantity')
                        if price * 100 != (price * 100).to_integral_value():
                            raise ValueError('price exceeds cent precision')
                        quantity, price_cents = int(qty), int(price * 100)
                    except (InvalidOperation, ValueError):
                        reason = 'invalid_number'
                    if reason is None:
                        try:
                            date = datetime.strptime(date_text, '%m/%d/%y %H:%M')
                        except ValueError:
                            reason = 'invalid_date'
                    if reason is None:
                        location = ADDRESS.fullmatch(address)
                        if not order_id.isdigit():
                            reason = 'invalid_order_id'
                        elif not location:
                            reason = 'invalid_address'
                if reason:
                    quality[reason] += 1
                    file_counts['rejected_rows'] += 1
                    rejected.append(dict(zip(FIELDS, values), source_file=path.name, source_row=line, rejection_reason=reason))
                    continue
                signature = tuple(values)
                duplicate = signature in seen
                seen.add(signature)
                quality['exact_duplicate_rows_retained'] += int(duplicate)
                quality['valid_rows'] += 1
                file_counts['valid_rows'] += 1
                rows.append(dict(order_id=order_id, product=product, quantity=quantity,
                                 price_cents=price_cents, sales_cents=quantity * price_cents,
                                 order_date=date.isoformat(sep=' '), year=date.year,
                                 month=date.month, hour=date.hour,
                                 city=f'{location[1].strip()} ({location[2]})',
                                 purchase_address=address, source_file=path.name,
                                 source_row=line, exact_duplicate=duplicate))
        manifest.append(dict(file=path.name, bytes=path.stat().st_size,
                             sha256=hashlib.sha256(path.read_bytes()).hexdigest(), **file_counts))
    quality['rejected_rows'] = len(rejected)
    assert quality['raw_rows'] == len(rows) + len(rejected)
    return rows, rejected, dict(quality), manifest


def summarize(rows, year=2019):
    """Một Order ID là một đơn; kiểm tra tính nhất quán trước khi tổng hợp."""
    selected = [r for r in rows if r['year'] == year]
    if not selected:
        raise ValueError(f'Không có dữ liệu hợp lệ cho năm {year}.')
    orders = defaultdict(list)
    for row in selected:
        orders[row['order_id']].append(row)
    conflicts = [oid for oid, items in orders.items()
                 if len({(r['order_date'], r['purchase_address']) for r in items}) != 1]
    if conflicts:
        raise ValueError(f'Order ID có thời gian/địa chỉ mâu thuẫn: {conflicts[:5]}')
    def aggregate(key, domain=None):
        groups = defaultdict(list)
        for row in selected:
            groups[row[key]].append(row)
        return [dict(label=value, sales_cents=sum(r['sales_cents'] for r in groups[value]),
                     quantity=sum(r['quantity'] for r in groups[value]),
                     orders=len({r['order_id'] for r in groups[value]}),
                     line_items=len(groups[value])) for value in (domain if domain is not None else sorted(groups))]
    monthly = aggregate('month', range(1, 13))
    cities = sorted(aggregate('city'), key=lambda r: (-r['sales_cents'], r['label']))
    hourly = aggregate('hour', range(24))
    products = sorted(aggregate('product'), key=lambda r: (-r['quantity'], r['label']))
    pair_counts, product_orders = Counter(), Counter()
    for items in orders.values():
        basket = sorted({r['product'] for r in items})
        product_orders.update(basket)
        pair_counts.update(itertools.combinations(basket, 2))
    pairs = [dict(product_a=a, product_b=b, orders=count,
                  support=count / len(orders),
                  confidence_a_to_b=count / product_orders[a],
                  confidence_b_to_a=count / product_orders[b],
                  lift=count * len(orders) / (product_orders[a] * product_orders[b]))
             for (a, b), count in sorted(pair_counts.items(), key=lambda kv: (-kv[1], kv[0]))]
    total = sum(r['sales_cents'] for r in selected)
    for grouping in (monthly, cities, hourly, products):
        assert sum(r['sales_cents'] for r in grouping) == total
    assert sum(r['orders'] for r in hourly) == len(orders)
    assert sum(r['orders'] for r in monthly) == len(orders)
    return dict(year=year, sales_cents=total, valid_line_items=len(selected),
                orders=len(orders), quantity=sum(r['quantity'] for r in selected),
                excluded_other_year_rows=len(rows)-len(selected),
                duplicate_rows_retained=sum(r['exact_duplicate'] for r in selected),
                aov_cents=total / len(orders), monthly=monthly, cities=cities,
                hourly=hourly, products=products, pairs=pairs)


def write_csv(path, rows, fieldnames=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = fieldnames or list(rows[0] if rows else [])
    if not fieldnames:
        return
    with path.open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
if not (DATA_DIR / 'sales2019_1.csv').exists():
    raise RuntimeError('Mở notebook từ thư mục gốc repo chứa data/sales2019_1.csv.')
import json
rows, rejected, quality, manifest = load_sales(DATA_DIR)
print(json.dumps(quality, indent=2, ensure_ascii=False))
print('Số file:', len(manifest))


{
  "raw_rows": 186850,
  "exact_duplicate_rows_retained": 264,
  "valid_rows": 185950,
  "empty_row": 545,
  "repeated_header": 355,
  "rejected_rows": 900
}
Số file: 12


## 2. Xác định phạm vi và chỉ tiêu
Chỉ chọn năm 2019 theo ngày đặt hàng thực tế. Tiền được tính bằng cent nguyên; số đơn là số Order ID duy nhất, không phải số dòng. Các dòng ngoài năm vẫn còn trong dữ liệu sạch.


In [2]:
summary = summarize(rows, year=2019)
print(f"Doanh thu: ${summary['sales_cents']/100:,.2f}")
print(f"Số đơn: {summary['orders']:,}")
print(f"Số lượng sản phẩm: {summary['quantity']:,}")
print(f"Dòng ngoài năm phân tích: {summary['excluded_other_year_rows']:,}")
assert sum(r['sales_cents'] for r in summary['monthly']) == summary['sales_cents']
assert sum(r['orders'] for r in summary['hourly']) == summary['orders']


Doanh thu: $34,483,365.68
Số đơn: 178,406
Số lượng sản phẩm: 209,038
Dòng ngoài năm phân tích: 34


## 3. Tạo bảng kết quả và báo cáo
Hàm `run` dưới đây tạo các bảng tổng hợp và báo cáo kết quả. Biểu đồ SVG được hiển thị ngay trong notebook; báo cáo `outputs/report.html` cũng mở được độc lập.


In [3]:
"""Báo cáo HTML độc lập và SVG, không cần internet hoặc thư viện ngoài."""
from html import escape
from pathlib import Path

NAVY, TEAL, GOLD = '#142b43', '#087f8c', '#d99a24'


def bars(title, labels, values, unit, horizontal=False):
    width, height = (1100, max(390, len(labels) * 38 + 100)) if horizontal else (1100, 430)
    left, top, right, bottom = (290, 65, 130, 45) if horizontal else (100, 65, 35, 85)
    plot_w, plot_h = width-left-right, height-top-bottom
    maximum = max(values, default=1) or 1
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}" role="img" aria-label="{escape(title)}">',
           '<rect width="100%" height="100%" fill="white"/>',
           '<style>text{font-family:Segoe UI,Arial,sans-serif;fill:#243c50;font-size:14px}.title{font-size:21px;font-weight:700}.muted{fill:#64748b;font-size:12px}</style>',
           f'<text x="24" y="30" class="title">{escape(title)}</text>',
           f'<text x="24" y="50" class="muted">{escape(unit)}</text>']
    def fmt(x):
        return f'{x/1e6:.2f}M' if x >= 1e6 else f'{x/1000:.1f}K' if x >= 1000 else f'{x:,.0f}'
    if horizontal:
        step = plot_h / max(1, len(labels))
        for i, (label, value) in enumerate(zip(labels, values)):
            y, length = top+i*step, plot_w*value/maximum
            out += [f'<text x="{left-12}" y="{y+19}" text-anchor="end">{escape(str(label))}</text>',
                    f'<rect x="{left}" y="{y}" width="{length}" height="25" rx="4" fill="{TEAL if i else GOLD}"/>',
                    f'<text x="{left+length+9}" y="{y+19}">{fmt(value)}</text>']
    else:
        for tick in range(5):
            v, y = maximum*tick/4, top+plot_h*(1-tick/4)
            out += [f'<path d="M{left},{y} H{width-right}" stroke="#e2e8f0"/>',
                    f'<text x="{left-12}" y="{y+5}" text-anchor="end" class="muted">{fmt(v)}</text>']
        step = plot_w / max(1, len(labels))
        for i, (label, value) in enumerate(zip(labels, values)):
            h = plot_h*value/maximum
            x, y = left+i*step+step*.16, top+plot_h-h
            out += [f'<rect x="{x}" y="{y}" width="{step*.68}" height="{h}" rx="3" fill="{GOLD if value == maximum else TEAL}"><title>{escape(str(label))}: {value:,.2f}</title></rect>',
                    f'<text x="{x+step*.34}" y="{top+plot_h+24}" text-anchor="middle">{escape(str(label))}</text>']
    return '\n'.join(out+['</svg>'])


def build_report(s, quality, output_dir: Path):
    charts_dir = output_dir / 'charts'
    charts_dir.mkdir(exist_ok=True)
    monthly, cities, hourly, products, pairs = (s[k] for k in ['monthly', 'cities', 'hourly', 'products', 'pairs'])
    definitions = [
        ('01_monthly', 'Doanh thu theo tháng', [f'T{r["label"]}' for r in monthly], [r['sales_cents']/100 for r in monthly], 'USD', False),
        ('02_cities', 'Doanh thu theo thành phố và bang', [r['label'] for r in cities], [r['sales_cents']/100 for r in cities], 'USD', True),
        ('03_hourly', 'Số đơn hàng theo giờ đặt hàng', [str(r['label']) for r in hourly], [r['orders'] for r in hourly], 'Số Order ID duy nhất; giờ địa phương theo dữ liệu nguồn', False),
        ('04_products', 'Sản phẩm bán chạy theo số lượng', [r['label'] for r in products], [r['quantity'] for r in products], 'Số sản phẩm bán ra', True),
        ('05_pairs', '10 cặp sản phẩm xuất hiện cùng đơn nhiều nhất', [r['product_a']+' + '+r['product_b'] for r in pairs[:10]], [r['orders'] for r in pairs[:10]], 'Số đơn; mỗi cặp chỉ tính một lần trong một đơn', True)]
    svgs = {}
    for name, title, labels, values, unit, horizontal in definitions:
        if name == '05_pairs':
            # Tên cặp dài: tách thành hai dòng bằng danh sách ở bảng bên dưới.
            labels = [f'Cặp {i+1}' for i in range(len(values))]
        svg = bars(title, labels, values, unit, horizontal)
        (charts_dir / f'{name}.svg').write_text(svg, encoding='utf-8')
        svgs[name] = svg
    best_month = max(monthly, key=lambda r: r['sales_cents'])
    peaks = sorted(hourly, key=lambda r: (-r['orders'], r['label']))[:3]
    pair_text = f"{pairs[0]['product_a']} + {pairs[0]['product_b']} ({pairs[0]['orders']:,} đơn)" if pairs else 'Không có cặp sản phẩm.'
    findings = [
        f"Tháng {best_month['label']} có doanh thu cao nhất: ${best_month['sales_cents']/100:,.2f} ({best_month['sales_cents']/s['sales_cents']:.1%} cả năm).",
        f"{cities[0]['label']} dẫn đầu với ${cities[0]['sales_cents']/100:,.2f}.",
        'Ba giờ có nhiều đơn nhất: '+', '.join(f"{r['label']:02d}:00 ({r['orders']:,} đơn)" for r in peaks)+'.',
        f'Cặp đứng đầu: {pair_text}.',
        f"{products[0]['label']} bán nhiều nhất: {products[0]['quantity']:,} sản phẩm. Giá và nhu cầu là giả thuyết cần kiểm chứng, chưa phải kết luận nhân quả."]
    def table(headers, rows):
        return '<div class="scroll"><table><thead><tr>'+''.join(f'<th>{escape(h)}</th>' for h in headers)+'</tr></thead><tbody>'+''.join('<tr>'+''.join(f'<td>{escape(str(v))}</td>' for v in row)+'</tr>' for row in rows)+'</tbody></table></div>'
    pair_table = table(['Cặp', 'Sản phẩm A', 'Sản phẩm B', 'Số đơn', 'Support', 'Confidence A→B', 'Lift'],
                       [(i+1,r['product_a'],r['product_b'],f"{r['orders']:,}",f"{r['support']:.2%}",f"{r['confidence_a_to_b']:.2%}",f"{r['lift']:.2f}") for i,r in enumerate(pairs[:10])])
    quality_table = table(['Chỉ tiêu', 'Số dòng'], [(k,f'{v:,}') for k,v in quality.items()])
    cards = [('DOANH THU', f"${s['sales_cents']/100:,.2f}"), ('ĐƠN HÀNG', f"{s['orders']:,}"),
             ('SẢN PHẨM', f"{s['quantity']:,}"), ('GIÁ TRỊ TB / ĐƠN', f"${s['aov_cents']/100:,.2f}")]
    sections = ''.join(f'<section><span class="eyebrow">PHÂN TÍCH {i+1:02d}</span>{svgs[name]}<p>{escape(findings[{3:4,4:3}.get(i,i)])}</p>'+(pair_table if i==4 else '')+'</section>' for i,(name,*_) in enumerate(definitions))
    page = '''<!DOCTYPE html><html lang="vi"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Sales Reporting | 2019</title><style>
    *{box-sizing:border-box}body{margin:0;background:#f0f4f6;color:#142b43;font:16px/1.6 'Segoe UI',Arial,sans-serif}main{max-width:1180px;margin:auto;padding:38px 24px}header{background:#142b43;color:white;padding:36px;border-radius:16px;margin-bottom:24px}h1{font-size:42px;line-height:1.1;margin:12px 0}header p{color:#c6d6e4;max-width:800px}.eyebrow{font-size:12px;letter-spacing:2px;font-weight:700;color:#087f8c}header .eyebrow{color:#76d3d7}.cards{display:grid;grid-template-columns:repeat(4,1fr);gap:16px}.card,section{background:white;border:1px solid #e0e7eb;border-radius:12px;padding:24px;margin-bottom:24px}.card small{color:#597185;font-size:11px;letter-spacing:1px}.card strong{display:block;font-size:27px;margin-top:10px}section svg{display:block;width:100%;height:auto;margin-top:10px}h2{font-size:24px;margin:0 0 14px}.scroll{overflow:auto}table{width:100%;border-collapse:collapse;font-size:14px}th{text-align:left;background:#edf5f6;color:#087f8c}th,td{padding:12px;border-bottom:1px solid #e5edf0}li{margin-bottom:10px}a{color:#087f8c}.note{border-left:4px solid #d99a24;background:#fff9ed;padding:16px}footer{color:#64748b;font-size:13px;padding:12px}@media(max-width:800px){.cards{grid-template-columns:repeat(2,1fr)}h1{font-size:32px}.card strong{font-size:21px}main{padding:20px 12px}section{padding:16px}}@media print{body{background:white}section{break-inside:avoid}header{print-color-adjust:exact}.cards{grid-template-columns:repeat(4,1fr)}}
    </style><main>'''
    page += f'<header><span class="eyebrow">PYTHON MINI PROJECT / BUSINESS ANALYTICS</span><h1>Sales Reporting {s["year"]}</h1><p>Phân tích doanh thu, hành vi đặt hàng và sản phẩm mua kèm từ 12 file CSV. Báo cáo được tạo trực tiếp bằng Python, mở được khi không có internet.</p></header>'
    page += '<div class="cards">'+''.join(f'<div class="card"><small>{title}</small><strong>{value}</strong></div>' for title,value in cards)+'</div>'
    page += '<section><h2>5 phát hiện chính</h2><ol>'+''.join(f'<li>{escape(f)}</li>' for f in findings)+'</ol></section>'+sections
    page += f'''<section><h2>Đề xuất hành động</h2><ul><li>Ưu tiên thử nghiệm kế hoạch hàng tồn kho và chiến dịch trước mùa cao điểm, theo doanh thu tháng.</li><li>Phân bổ thử ngân sách theo thành phố; bổ sung dân số, chi phí và tỷ lệ chuyển đổi trước khi kết luận hiệu quả thị trường.</li><li>Thử A/B lịch quảng cáo quanh giờ đặt hàng cao điểm; dữ liệu đơn hàng chưa chứng minh khung giờ quảng cáo tối ưu.</li><li>Thử gói sản phẩm mua kèm có support đủ lớn; theo dõi lợi nhuận và kết quả thử nghiệm.</li></ul></section>
    <section><h2>Chất lượng dữ liệu và quy tắc tính</h2>{quality_table}<p class="note">Giữ nguyên các dòng trùng hoàn toàn và đánh dấu, vì dữ liệu không có khóa dòng đơn để xác nhận đó là lỗi. Có {s['excluded_other_year_rows']:,} dòng hợp lệ ngoài năm {s['year']}, được giữ trong sales_cleaned.csv và loại khỏi các chỉ tiêu năm.</p><ul><li>Đơn vị tiền giả định là USD theo notebook nguồn. Doanh thu = số lượng × đơn giá, tính bằng số nguyên cent; chưa có chi phí, thuế, giảm giá hoặc hoàn trả.</li><li>Một dòng là một dòng sản phẩm; số đơn = số Order ID duy nhất. AOV = doanh thu / số đơn.</li><li>Tháng lấy từ ngày đặt hàng thực tế, không lấy từ tên file. Thành phố đi kèm bang.</li><li>Giờ giữ nguyên theo dữ liệu nguồn, không quy đổi múi giờ vì chưa có thông tin múi giờ.</li><li>Mỗi cặp sản phẩm được tính tối đa một lần/đơn. Support = số đơn có cả A,B / tổng đơn; confidence A→B = số đơn có A,B / số đơn có A; lift = confidence / tỷ lệ đơn có B.</li></ul></section>
    <footer>Nguồn tham khảo: <a href="https://github.com/boringPpl/Sales-Reporting">Sales-Reporting</a>. Các bảng tổng hợp được lưu tại outputs/tables; doanh thu trong CSV sử dụng đơn vị cent.</footer></main></html>'''
    (output_dir / 'report.html').write_text(page, encoding='utf-8')
    markdown = '# Báo cáo phân tích bán hàng '+str(s['year'])+'\n\n'+'\n'.join(f'{i+1}. {f}' for i,f in enumerate(findings))
    markdown += f"\n\nTổng doanh thu: ${s['sales_cents']/100:,.2f}. Số đơn: {s['orders']:,}. Số sản phẩm: {s['quantity']:,}.\n\n"
    markdown += 'Giữ các dòng trùng và đánh dấu vì chưa xác nhận được khóa dòng đơn. Loại các ngày ngoài năm phân tích khỏi KPI. Đây là phân tích mô tả; khuyến nghị quảng cáo và nguyên nhân bán chạy cần thử nghiệm. Xem report.html để đọc đầy đủ biểu đồ và quy tắc.\n'
    (output_dir / 'BAO_CAO.md').write_text(markdown, encoding='utf-8')

def run(data_dir=ROOT / 'data', output_dir=ROOT / 'outputs', year=2019):
    data_dir, output_dir = Path(data_dir), Path(output_dir)
    rows, rejected, quality, manifest = load_sales(data_dir)
    result = summarize(rows, year)
    output_dir.mkdir(parents=True, exist_ok=True)
    write_csv(output_dir / 'data' / 'sales_cleaned.csv', rows)
    write_csv(output_dir / 'data' / 'rejected_rows.csv', rejected, FIELDS + ['source_file', 'source_row', 'rejection_reason'])
    write_csv(output_dir / 'data' / 'input_manifest.csv', manifest,
              ['file', 'bytes', 'sha256', 'raw_rows', 'valid_rows', 'rejected_rows'])
    for name in ['monthly', 'cities', 'hourly', 'products', 'pairs']:
        write_csv(output_dir / 'tables' / f'{name}.csv', result[name],
                  ['product_a', 'product_b', 'orders', 'support', 'confidence_a_to_b', 'confidence_b_to_a', 'lift'] if name == 'pairs' else None)
    (output_dir / 'data_quality.json').write_text(json.dumps(quality, indent=2), encoding='utf-8')
    (output_dir / 'summary.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    build_report(result, quality, output_dir)
    print(f"Revenue {year}: ${result['sales_cents']/100:,.2f} | Orders: {result['orders']:,}")
    print('Báo cáo đã được tạo tại outputs/report.html')
    return result, quality



summary, quality = run()

class Chart:
    def __init__(self, name):
        self.svg = (ROOT / 'outputs' / 'charts' / name).read_text(encoding='utf-8')
    def _repr_html_(self):
        return self.svg
    def __repr__(self):
        return 'Biểu đồ SVG của báo cáo Sales Reporting'


Revenue 2019: $34,483,365.68 | Orders: 178,406
Báo cáo đã được tạo tại outputs/report.html


## 4. Tháng nào có doanh thu cao nhất?
Lấy năm và tháng từ ngày đặt hàng để tránh gộp các ngày tháng 1/2020 vào tháng 1/2019.


In [4]:
best = max(summary['monthly'], key=lambda r: r['sales_cents'])
print(f"Tháng {best['label']}: ${best['sales_cents']/100:,.2f}")
Chart('01_monthly.svg')


Tháng 12: $4,613,443.34


Biểu đồ SVG của báo cáo Sales Reporting

## 5. Thành phố nào có doanh thu cao nhất?
Tách thành phố theo bang để giữ riêng Portland (OR) và Portland (ME). Doanh thu cao chưa có nghĩa thị trường có ROI cao.


In [5]:
best = summary['cities'][0]
print(best['label'], f"${best['sales_cents']/100:,.2f}")
Chart('02_cities.svg')


San Francisco (CA) $8,259,719.03


Biểu đồ SVG của báo cáo Sales Reporting

## 6. Giờ nào có nhiều đơn hàng nhất?
Đếm đơn duy nhất, không đếm dòng sản phẩm. Dữ liệu chưa có múi giờ và kết quả quảng cáo, nên dùng để đề xuất thử nghiệm A/B.


In [6]:
for row in sorted(summary['hourly'], key=lambda r: (-r['orders'], r['label']))[:3]:
    print(f"{row['label']:02d}:00 — {row['orders']:,} đơn")
Chart('03_hourly.svg')


19:00 — 12,377 đơn
12:00 — 12,082 đơn
11:00 — 11,882 đơn


Biểu đồ SVG của báo cáo Sales Reporting

## 7. Sản phẩm nào thường mua cùng nhau?
Dùng tập sản phẩm của mỗi đơn rồi tạo mọi tổ hợp 2 phần tử; A+B+C đóng góp A+B, A+C, B+C. Support là tỷ lệ đơn chứa cả hai. Lift so sánh với kỳ vọng độc lập.


In [7]:
for row in summary['pairs'][:5]:
    print(row['product_a'], '+', row['product_b'], '|', row['orders'], 'đơn',
          '| support', f"{row['support']:.2%}", '| lift', f"{row['lift']:.2f}")
Chart('05_pairs.svg')


Lightning Charging Cable + iPhone | 1010 đơn | support 0.57% | lift 1.22
Google Phone + USB-C Charging Cable | 997 đơn | support 0.56% | lift 1.47
Wired Headphones + iPhone | 462 đơn | support 0.26% | lift 0.64
Google Phone + Wired Headphones | 422 đơn | support 0.24% | lift 0.72
Apple Airpods Headphones + iPhone | 372 đơn | support 0.21% | lift 0.63


Biểu đồ SVG của báo cáo Sales Reporting

## 8. Sản phẩm nào bán nhiều nhất?
Xếp theo tổng số lượng bán ra. Giải thích vì giá thấp hoặc nhu cầu cao cần dữ liệu và kiểm chứng bổ sung.


In [8]:
for row in summary['products'][:5]:
    print(row['label'], '|', f"{row['quantity']:,}", 'sản phẩm')
Chart('04_products.svg')


AAA Batteries (4-pack) | 31,012 sản phẩm
AA Batteries (4-pack) | 27,635 sản phẩm
USB-C Charging Cable | 23,971 sản phẩm
Lightning Charging Cable | 23,211 sản phẩm
Wired Headphones | 20,553 sản phẩm


Biểu đồ SVG của báo cáo Sales Reporting

## 9. Kết luận và hướng phát triển

- Dùng tháng cao điểm và thành phố dẫn đầu để đề xuất kế hoạch hàng hóa; bổ sung chi phí trước khi phân bổ ngân sách.
- Thử nghiệm lịch quảng cáo quanh giờ có nhiều đơn; thử gói sản phẩm mua kèm và đo lợi nhuận tăng thêm.
- Không có mã khách hàng, nên không thể tính tỷ lệ khách quay lại đáng tin cậy.
- Kiểm tra bổ sung: nếu chủ dữ liệu xác nhận dòng trùng là lỗi, so sánh KPI trước/sau loại trùng và ghi rõ thay đổi chính sách.
